Quick links
- API key: https://api.liquipedia.net/apikey
- Documentation: https://api.liquipedia.net/documentation/api/v3

## Setup

In [ ]:
import json
import time

import requests
import pandas as pd

In [ ]:
with open("secrets.json") as f:
    secrets = json.load(f)

In [ ]:
headers = {
    "User-Agent": "EsportUsernames/1.0 (https://github.com/hollowscene)",
    "authorization": "Apikey " + secrets.get("API_KEY"),
}

In [ ]:
with open("wiki_list/main_wiki_list.json", "r") as f:
    main_wiki_list = json.load(f)

main_wikis_param = "|".join(main_wiki_list.keys())

In [ ]:
def query(
    params: dict,
    url: str = "https://api.liquipedia.net/api/v3/player",
    headers: dict = headers,
    return_as_df: bool = True,
) -> list[dict] | pd.DataFrame:
    """Make a simple query to the Liquipedia API."""
    response = requests.get(url, headers=headers, params=params)

    status_code = response.status_code

    if status_code != 200:
        raise Exception(f"API call failed: {status_code} {response.text}.")

    data = response.json()["result"]

    if return_as_df:
        df = pd.DataFrame.from_dict(data)
        return df
    else:
        return data

## Explore

In [ ]:
params = {
    "wiki": main_wikis_param,
    "conditions": f"[[id::Lucky]]",
}

query(params)

In [ ]:
params = {
    "wiki": main_wikis_param,
    "query": "id, count::pageid",
    "order": "count::pageid DESC",
    "groupby": "id ASC",
    "limit": 1000,
}

query(params)

In [ ]:
params = {
    "wiki": main_wikis_param,
    "query": "id, count::pageid",
    "order": "count::pageid DESC",
    "groupby": "id ASC",
    "limit": 1000,
}

query(params, return_as_df=False)

### Pagination (TODO)

In [ ]:
def paginated_query(
    params: dict,
    url: str = "https://api.liquipedia.net/api/v3/player",
    headers: dict = headers,
    wait: int = 30,
    return_as_df: bool = True,
) -> list[dict] | pd.DataFrame:
    """Make a paginated query to the Liquipedia API.

    Use this function for queries that have more than 1,000 rows. It will
    automatically query the next 1,000 rows until no rows are returned.
    """
    assert "limit" not in params, "Limit parameter will be ignored for paginated queries."
    assert "offset" not in params, "Offset parameter will be ignored for paginated queries."

    params["limit"] = 1000
    params["offset"] = 0

    iter_n = 1
    data_iter = query(params=params, url=url, headers=headers, return_as_df=False)
    data = data_iter
    print(f"Iteration {iter_n}: {len(data):,} total rows queried.")

    while len(data_iter) >= 1000:
        time.sleep(wait)
        iter_n += 1
        params["offset"] = params["offset"] + 1000
        data_iter = query(params=params, url=url, headers=headers, return_as_df=False)
        data.extend(data_iter)
        print(f"Iteration {iter_n:,}: {len(data):,} total rows queried.")

    print(f"Finished pagination query after {iter_n:,} iterations with {len(data):,} total rows queried.")

    if return_as_df:
        df = pd.DataFrame.from_dict(data)
        return df
    else:
        return data

In [ ]:
params = {
    "wiki": main_wikis_param,
    "query": "id, count::pageid",
    "order": "count::pageid DESC",
    "groupby": "id ASC",
}

df = paginated_query(params)

In [ ]:
df_players = df.groupby("id", as_index=False)[["count_pageid"]].sum()
df_players.sort_values(by="count_pageid", ascending=False)

In [ ]:
params = {
    "wiki": main_wikis_param,
    "query": "count::pageid",
}

df_total_count = query(params)

In [ ]:
df_total_count["count_pageid"].sum()

In [ ]:
df_players["count_pageid"].sum()

In [ ]:
df_players.to_csv("data/players_usernames.csv", index=False)